# 7. 现代卷积神经网络

在本章中的每一个模型都曾一度占据主导地位，其中许多模型都是ImageNet竞赛的优胜者。

> ImageNet竞赛自2010年以来，一直是计算机视觉中监督学习进展的指向标。

这些模型包括：

*   **AlexNet**。它是第一个在大规模视觉竞赛中击败传统计算机视觉模型的大型神经网络；

*   **使用重复块的网络（VGG）**。它利用许多重复的神经网络块；

*   **网络中的网络（NiN）**。它重复使用由卷积层和 $1 \times 1$ 卷积层（用来代替全连接层）来构建深层网络;

*   **含并行连结的网络（GoogLeNet）**。它使用并行连结的网络，通过不同窗口大小的卷积层和最大汇聚层来并行抽取信息；

*   **残差网络（ResNet）**。它通过残差块构建跨层的数据通道，是计算机视觉中最流行的体系架构；

*   **稠密连接网络（DenseNet）**。它的计算成本很高，但给我们带来了更好的效果。

虽然深度神经网络的概念非常简单——将神经网络堆叠在一起。但由于不同的网络架构和超参数选择，这些神经网络的性能会发生很大变化。本章介绍的神经网络是将人类直觉和相关数学见解结合后，经过大量研究试错后的结晶。 

---


## 7.1 深度卷积神经网络 (AlexNet)

> 如果说 LeNet 是神经网络的“童年”，那么 **AlexNet** 就是神经网络的“成年礼”。2012 年，AlexNet 在 ImageNet 竞赛中以绝对优势夺冠，彻底终结了传统计算机视觉的统治，开启了深度学习的黄金时代。

### 1. 核心动机：为什么 LeNet 不够了？

LeNet 在处理手写数字（28x28）时非常完美，但在面对真实世界的自然图像（如 ImageNet 的高清图）时遇到了瓶颈：

1.  **特征复杂度**：真实图像的纹理、光照和背景极其复杂，LeNet 的参数量不足以捕捉这些特征。
2.  **计算资源**：2012 年左右，GPU 算力开始爆发，使得训练更深、更宽的网络成为可能。
3.  **激活函数限制**：Sigmoid 在深层网络中极易导致**梯度消失**。

---

### 2. AlexNet 的五大核心创新

1.  **更深的网络结构**：8 层加权层（5 层卷积 + 3 层全连接）。
2.  **使用 ReLU 激活函数**：用 $\text{max}(0, x)$ 代替 Sigmoid。ReLU 在正区间导数为 1，彻底解决了深层网络的梯度消失问题，且计算速度极快。
3.  **使用 Dropout (暂退法)**：在最后的全连接层引入 Dropout（我们在 4.6 节学过），有效控制了巨大参数量带来的过拟合。
4.  **数据增广 (Data Augmentation)**：通过随机裁剪、翻转和亮度调整，人为增加训练样本，提高模型鲁棒性。
5.  **最大汇聚 (Max Pooling)**：用最大汇聚代替 LeNet 的平均汇聚，更能突出图像的显著特征。

---

### 3. 构建 AlexNet

AlexNet 的输入通常是 $224 \times 224$。由于 Fashion-MNIST 只有 $28 \times 28$，我们在加载数据时需要将其**强制放大**。


In [2]:
import torch
from torch import nn

def get_alexnet_model(num_classes: int = 10) -> nn.Sequential:
    """构建 AlexNet 模型。通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
    
    结构参考 2012 年经典论文，针对 Fashion-MNIST 输出 10 类。
    """
    net = nn.Sequential(
        # 第一块：大尺寸卷积核提取粗略特征
        # 输入：(Batch, 1, 224, 224)
        nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2),  # -> (Batch, 64, 55, 55)
        nn.ReLU(),
        # 重叠汇聚：
        #   - 缓解过拟合;
        #   - 增大感受野，并避免信息通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。丢失过快
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 64, 27, 27)

        # 第二块：减小核尺寸，增加通道数
        # 输入：(Batch, 64, 27, 27)
        nn.Conv2d(64, 192, kernel_size=5, padding=2),           # -> (Batch, 192, 27, 27)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 192, 13, 13)

        # 第三块：连续三个卷积层提取精细特征
        # 输入：(Batch, 192, 13, 13)
        nn.Conv2d(192, 384, kernel_size=3, padding=1),          # -> (Batch, 384, 13, 13)
        nn.ReLU(),
        nn.Conv2d(384, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 256, 6, 6)

        # 第四块：全连接层 + Dropout
        nn.Flatten(),                                           # -> (Batch, 256 * 6 * 6)
        # 参数量主要集中在该全连接层
        nn.Linear(256 * 6 * 6, 4096),                           # -> (Batch, 4096)
        nn.ReLU(),
        # Dropout 抑制过拟合
        nn.Dropout(p=0.5),
        # 4096 -> 4096 是当时认为的力大砖飞，
        # 现在更倾向于通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
        nn.Linear(4096, 4096),                                  # -> (Batch, 4096)
        nn.ReLU(),
        nn.Dropout(p=0.5),

        # 输出层
        nn.Linear(4096, num_classes)                            # -> (Batch, num_classes)
    )
    return net

# --- 验证维度流转 ---
X = torch.randn(1, 1, 224, 224)
net = get_alexnet_model()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:15} 输出形状: {X.shape}")

Conv2d          输出形状: torch.Size([1, 64, 55, 55])
ReLU            输出形状: torch.Size([1, 64, 55, 55])
MaxPool2d       输出形状: torch.Size([1, 64, 27, 27])
Conv2d          输出形状: torch.Size([1, 192, 27, 27])
ReLU            输出形状: torch.Size([1, 192, 27, 27])
MaxPool2d       输出形状: torch.Size([1, 192, 13, 13])
Conv2d          输出形状: torch.Size([1, 384, 13, 13])
ReLU            输出形状: torch.Size([1, 384, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
MaxPool2d       输出形状: torch.Size([1, 256, 6, 6])
Flatten         输出形状: torch.Size([1, 9216])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([

---

### 4. 训练 AlexNet

> 由于训练 ImageNet 模型花费时间较长，因而，仅出于演示目的，我们使用 `Fashion-MNIST` 进行对 AlexNet 的训练。

为了适配 AlexNet，我们需要修改之前的 `load_fashion_mnist` 函数，加入 `Resize` 步骤。


In [3]:
import d2l_utils as d2l
from torch.utils import data

def load_data_alexnet(batch_size: int, resize: int = 224) -> tuple[data.dataloader, data.dataloader]:
    """加载 Fashion-MNIST 并调整尺寸以适配 AlexNet。"""
    return d2l.load_fashion_mnist(batch_size=batch_size, resize=resize)

# 建议配置
batch_size = 128
train_iter, test_iter = load_data_alexnet(batch_size)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data


In [4]:
import d2l_utils as d2l
def train_alexnet(
    net: nn.Module,
    train_iter: data.DataLoader,
    test_iter: data.DataLoader,
    num_epochs: int,
    lr: float,
    device: torch.device
) -> None:
    """训练函数，用以演示 AlexNet 模型的训练过程。"""

    # 1. 初始化权重、搬运到 GPU
    def init_weights(m: nn.Module):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            # 对输出层使用 Xavier 初始化（如果不接 ReLU）
            # 注意：这里的逻辑依赖于 net 的结构定义，如果最后一层是 net[-1]
            if m == net[-1]:
                nn.init.xavier_uniform_(m.weight)
            else:
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    net.apply(init_weights)
    net.to(device)

    # 2. 优化器与损失函数
    # 按照论文配置 momentum 和 weight_decay
    optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
    loss_fn = nn.CrossEntropyLoss()

    print(f"在设备 {device} 上启动训练...")

    # 3. 训练循环
    for epoch in range(num_epochs):
        net.train()
        metric = d2l.Accumulator(3, device) # [train_loss_sum, train_acc_sum, n]

        for X, y in train_iter:
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            
            y_hat = net(X)
            l = loss_fn(y_hat, y)
            l.backward()
            optimizer.step()

            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.count_correct_tensor(y_hat, y), X.shape[0])

        # 评估测试集
        train_loss = (metric[0]/metric[2]).item()
        train_acc = (metric[1]/metric[2]).item()
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter, device)
        print(f"Epoch {epoch+1} | Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}")

# 运行示例
batch_size = 128 # 如果你的显卡显存小于 8GB，建议将 batch_size 设为 64 或 32
lr, num_epochs = 0.01, 10 # 建议先用较小的学习率观察收敛情况
device = d2l.get_default_device()

train_iter, test_iter = load_data_alexnet(batch_size)

net = get_alexnet_model()

train_alexnet(net, train_iter, test_iter, num_epochs, lr, device)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
Epoch 1 | Loss: 0.575 | Train Acc: 0.789 | Test Acc: 0.865
Epoch 2 | Loss: 0.320 | Train Acc: 0.883 | Test Acc: 0.884
Epoch 3 | Loss: 0.272 | Train Acc: 0.901 | Test Acc: 0.902
Epoch 4 | Loss: 0.243 | Train Acc: 0.911 | Test Acc: 0.914
Epoch 5 | Loss: 0.220 | Train Acc: 0.920 | Test Acc: 0.910
Epoch 6 | Loss: 0.204 | Train Acc: 0.925 | Test Acc: 0.915
Epoch 7 | Loss: 0.182 | Train Acc: 0.933 | Test Acc: 0.919
Epoch 8 | Loss: 0.173 | Train Acc: 0.937 | Test Acc: 0.920
Epoch 9 | Loss: 0.159 | Train Acc: 0.942 | Test Acc: 0.923
Epoch 10 | Loss: 0.148 | Train Acc: 0.947 | Test Acc: 0.922


---

### 5. 细致梳理：AlexNet 的数学与逻辑

1.  **计算量与参数量**：
    *   AlexNet 虽然只有 8 层，但参数量高达 **6000 万**个。
    *   绝大部分参数集中在第一个全连接层：$256 \times 6 \times 6 \times 4096 \approx 3700$ 万。
2.  **ReLU 的胜利**：
    *   ReLU 的计算就是一行 `if x > 0`，而 Sigmoid 涉及复杂的指数运算。这使得 AlexNet 的训练速度比同规模的 Sigmoid 网络快 6 倍以上。
3.  **为什么第一层用 11x11 的大核？**
    *   在输入图片很大（224x224）时，需要一个足够大的窗口来捕捉最初的物体轮廓。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **显存预警**：AlexNet 的全连接层非常吃显存。如果你的显卡显存较小（如 4GB），请务必调小 `batch_size`（如 64 或 32）。
*   **学习率调优**：由于 AlexNet 比较深，建议初始学习率设小一点（如 0.01），并配合 Adam 优化器。

---

### 7. 总结 Checklist

*   **ReLU**：明白它为什么能取代 Sigmoid（解决梯度消失）。
*   **Dropout**：明白它在 AlexNet 这种大参数模型中的必要性。
*   **Resize**：掌握如何处理输入尺寸不匹配的问题。
*   **架构感**：记住 AlexNet “卷积-卷积-卷积卷积卷积-全连接”的节奏。

---


## 7.2 使用块的网络（VGG）
